# PhysAR-IDS -- NB13 (v6): computational cost -- latency, throughput, size, training time

**Why a rerun.** The v4 chapter quoted 6.26 ms for XGBoost alone and 13.2 ms mean / 53.1 ms P99 for the gated pipeline, measured on a Colab T4 instance, and mapped them onto a Tier 1 (< 10 ms) / Tier 2 (< 100 ms) requirement. Those numbers depended on the API path (the scikit-learn wrapper adds a lot of per-call overhead), on thread settings, and on a cloud machine that says little about a substation computer. This notebook measures every component on **this machine**, discloses the hardware, and separates what is inherent to the model from what is wrapper overhead.

**What is measured (5 seeds; models differ per seed, timings are repeated per seed):**
* **Components:** the standard tuned XGBoost (refitted per seed, as in NB2/NB5/NB8) and the NB5b robust XGBoost, each through the scikit-learn API and through the lean `Booster.inplace_predict` path; the NB6 autoencoder gate (5 AEs, 93-32-12-32-93, GELU) in PyTorch and in plain NumPy; the serial **pipeline** (gate, then the robust classifier only for unflagged rows) in both variants; TreeSHAP for one row; split-conformal set construction (NB11) and the 6-model CV+ set construction (NB11b); the federated MLP (NB12) in PyTorch and NumPy.
* **Single-row latency** (the online case): 2,000 test rows, each timed individually with `perf_counter_ns` after 200 warm-up calls, garbage collection off, at **1 thread** (an edge device) and at **all threads**. Mean, median, P95, P99, max, share of calls under 10 ms and 100 ms, bootstrap 95 % intervals (resampling calls; consecutive calls are not independent, so the intervals are indicative), and three consecutive blocks per run to expose drift.
* **Batch throughput** on CPU (all threads) and on the GPU if present, for batches of 64 to 65,536 rows.
* **Size and complexity:** model files, tree and node counts, parameters, analytic multiply-adds, memory growth when loading.
* **Training time:** XGBoost fit on the GPU (every seed) and on the CPU (first seed only), and the AE gate (5 folds x 30 epochs, the NB7 loop) on this device plus one CPU fold. **Not included:** the NB5b adversarial-training cost (attack generation); take that from the NB5b log.
* **Correctness checks:** the NumPy gate must reproduce the PyTorch gate; the lean and sklearn XGBoost paths must give the same predictions; the CV+ and split-conformal timing uses real models but placeholder calibration arrays of the true size (the values do not change the cost).

**Reading the results.** The Tier bounds are the thesis's own assumed requirements, not a standard. A P99 under a bound on this machine says nothing about another machine; the hardware table is part of the result. Latency numbers include Python call overhead on purpose (that is what a Python deployment pays); the throughput table shows the per-row cost when that overhead is amortised.

**Needs:** `phase1_data_v6.pkl`, `xgb_tuned_best_params_v6.json`, `xgb_robust_v6_seed{s}.json` (NB5b), `ae_gate_v6_seed{s}.pt` (NB6), `surrogate_mlp` not needed. **Run this notebook with nothing else heavy running.** Runtime: about 30-45 min.

## 0. Setup

In [ ]:
import os, sys, random, time, json, warnings, math, gc, platform, tempfile
import numpy as np, pandas as pd, joblib
import torch, torch.nn as nn, torch.nn.functional as F
import xgboost as xgb
from scipy import stats
from scipy.optimize import minimize_scalar
from scipy.special import erf
from sklearn.model_selection import KFold
import matplotlib; import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = False
N_BOOT = 2000
N_SINGLE = 2000; N_WARM = 200; N_BLOCKS = 3
TIER1_MS = 10.0; TIER2_MS = 100.0
BATCHES = [64, 1024, 16384, 65536]
GATE_HIDDEN = 32; GATE_BOTTLENECK = 12; GATE_EPOCHS = 30; N_FOLDS = 5
N_CV = 6; N_CAL = 84000

os.environ['PYTHONHASHSEED'] = '42'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
print(f'torch {torch.__version__} ({DEVICE}) | xgboost {xgb.__version__} backend: {"cpu(hist)" if _cpu else "gpu(cuda,hist)"}')
print('Working directory:', os.getcwd())

In [ ]:
try: import psutil
except Exception: psutil = None
try: from threadpoolctl import threadpool_limits
except Exception: threadpool_limits = None
def cpu_name():
    try:
        if sys.platform.startswith('win'):
            import subprocess; return subprocess.check_output('wmic cpu get name', shell=True).decode().split('\n')[1].strip()
        for l in open('/proc/cpuinfo'):
            if l.startswith('model name'): return l.split(':')[1].strip()
    except Exception: pass
    return platform.processor() or 'unknown'
N_ALL = os.cpu_count() or 1
HW = {'cpu': cpu_name(), 'logical cores': N_ALL, 'physical cores': (psutil.cpu_count(logical=False) if psutil else 'n/a'), 'RAM GB': (round(psutil.virtual_memory().total / 1e9, 1) if psutil else 'n/a'),
      'OS': platform.platform(), 'python': platform.python_version(), 'numpy': np.__version__, 'torch': torch.__version__, 'xgboost': xgb.__version__, 'GPU': (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'),
      'threadpoolctl': threadpool_limits is not None, 'psutil': psutil is not None}
print(pd.Series(HW).to_string())
if N_ALL < 2: print('NOTE: one logical core -- the "all threads" setting equals the 1-thread setting')

## 1. Load v6 data and models

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']).astype(np.int64); y_test = np.asarray(data['y_test']).astype(np.int64); N_FEAT = X_train.shape[1]
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
for k_ in ('n_estimators', 'max_depth', 'min_child_weight'):
    if k_ in best_params: best_params[k_] = int(best_params[k_])
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)
rs = np.random.RandomState(13); sel = rs.choice(len(X_test), size=min(N_SINGLE, len(X_test)), replace=False)
Xs = [np.ascontiguousarray(X_test[i:i + 1]) for i in sel]; Xs_t = [torch.from_numpy(x) for x in Xs]; Xs_1d = [x[0] for x in Xs]
print(f'train {len(y_train):,} | test {len(y_test):,} | single-row sample {len(Xs)} | max depth {best_params.get("max_depth")}, trees per class {best_params.get("n_estimators")}')

class AE(nn.Module):
    def __init__(self, n_in, bottleneck=GATE_BOTTLENECK, hidden=GATE_HIDDEN):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_in, hidden), nn.GELU(), nn.Linear(hidden, bottleneck), nn.GELU())
        self.decoder = nn.Sequential(nn.Linear(bottleneck, hidden), nn.GELU(), nn.Linear(hidden, n_in))
    def forward(self, x): return self.decoder(self.encoder(x))
class SurrogateMLP(nn.Module):
    def __init__(self, n_features, n_classes=6, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_features, hidden), nn.ReLU(), nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, n_classes))
    def forward(self, x): return self.net(x)
def gelu_np(x): return 0.5 * x * (1.0 + erf(x / np.sqrt(2.0)))
class GateNP:
    # the AE ensemble in plain NumPy: weights stacked over the 5 models
    def __init__(self, models, tau):
        g = lambda f: np.stack([f(m).detach().cpu().numpy().astype(np.float32) for m in models])
        self.W1, self.b1 = g(lambda m: m.encoder[0].weight), g(lambda m: m.encoder[0].bias); self.W2, self.b2 = g(lambda m: m.encoder[2].weight), g(lambda m: m.encoder[2].bias)
        self.W3, self.b3 = g(lambda m: m.decoder[0].weight), g(lambda m: m.decoder[0].bias); self.W4, self.b4 = g(lambda m: m.decoder[2].weight), g(lambda m: m.decoder[2].bias); self.tau = float(tau)
    def err1(self, x):                                     # x: (93,) float32
        h = gelu_np(self.W1 @ x + self.b1); z = gelu_np(np.einsum('mbh,mh->mb', self.W2, h) + self.b2); d = gelu_np(np.einsum('mhb,mb->mh', self.W3, z) + self.b3)
        o = np.einsum('mih,mh->mi', self.W4, d) + self.b4; return float(((o - x) ** 2).mean(1).mean())
    def err_batch(self, X):
        h = gelu_np(np.einsum('mhi,ni->mnh', self.W1, X) + self.b1[:, None]); z = gelu_np(np.einsum('mbh,mnh->mnb', self.W2, h) + self.b2[:, None]); d = gelu_np(np.einsum('mhb,mnb->mnh', self.W3, z) + self.b3[:, None])
        o = np.einsum('mih,mnh->mni', self.W4, d) + self.b4[:, None]; return ((o - X[None]) ** 2).mean(2).mean(0)
class MlpNP:
    def __init__(self, m):
        L = [l for l in m.net if isinstance(l, nn.Linear)]; self.W = [l.weight.detach().cpu().numpy().astype(np.float32) for l in L]; self.b = [l.bias.detach().cpu().numpy().astype(np.float32) for l in L]
    def pred1(self, x):
        h = np.maximum(self.W[0] @ x + self.b[0], 0); h = np.maximum(self.W[1] @ h + self.b[1], 0); return int(np.argmax(self.W[2] @ h + self.b[2]))
def gate_torch_err(models, xt):
    with torch.no_grad(): return float(torch.stack([((m(xt) - xt) ** 2).mean(1) for m in models]).mean(0))
def gate_torch_batch(X):
    xt = torch.from_numpy(X)
    with torch.no_grad(): return torch.stack([((m(xt) - xt) ** 2).mean(1) for m in GATE_CUR]).mean(0)
def load_gate(seed):
    ck = torch.load(f'ae_gate_{PIPELINE_VERSION}_seed{seed}.pt', map_location='cpu', weights_only=False); ms_ = []
    for st in ck['state']: m = AE(N_FEAT); m.load_state_dict(st); m.eval(); ms_.append(m)
    return ms_, float(ck['tau'])
def rss_mb(): return psutil.Process().memory_info().rss / 1e6 if psutil else float('nan')
print('classes defined')

## 2. Timing machinery

In [ ]:
def time_calls(fn, xs, warm=N_WARM):
    for i in range(warm): fn(xs[i % len(xs)])
    gc.collect(); gc.disable(); t = np.empty(len(xs))
    try:
        for i, x in enumerate(xs):
            t0 = time.perf_counter_ns(); fn(x); t[i] = time.perf_counter_ns() - t0
    finally: gc.enable()
    return t / 1e6                                                                    # ms
def lat_stats(t, rng):
    bs = rng.randint(0, len(t), size=(N_BOOT // 4, len(t))); tb = t[bs]; med_b = np.median(tb, 1); p99_b = np.percentile(tb, 99, axis=1); blocks = np.array_split(t, N_BLOCKS)
    return {'mean': float(t.mean()), 'median': float(np.median(t)), 'p95': float(np.percentile(t, 95)), 'p99': float(np.percentile(t, 99)), 'max': float(t.max()), 'under10': float((t < TIER1_MS).mean()), 'under100': float((t < TIER2_MS).mean()),
            'median_ci': np.percentile(med_b, [2.5, 97.5]), 'p99_ci': np.percentile(p99_b, [2.5, 97.5]), 'block_medians': np.array([np.median(b) for b in blocks])}
class limit_threads:
    def __init__(self, k): self.k = k
    def __enter__(self):
        self.prev = torch.get_num_threads(); torch.set_num_threads(self.k); self.ctx = threadpool_limits(self.k) if threadpool_limits else None
        if self.ctx: self.ctx.__enter__()
    def __exit__(self, *a):
        torch.set_num_threads(self.prev)
        if self.ctx: self.ctx.__exit__(*a)
def set_xgb_threads(clf, k):
    clf.set_params(n_jobs=k); clf.get_booster().set_param({'nthread': k})
def cpu_copy(path):
    c = xgb.XGBClassifier(); c.load_model(path); c.set_params(device='cpu', n_jobs=1); c.get_booster().set_param({'device': 'cpu', 'nthread': 1}); return c
def pad_ms(x): return f'{x:.3f}'
print('timing machinery defined')

## 3. Per-seed measurements

For each seed: refit the standard model (GPU fit time recorded), load the robust model and the gate, check that the NumPy and lean paths reproduce the reference ones, then time every component at 1 thread and at all threads.

In [ ]:
THREADS = sorted({1, N_ALL}); RES = {}; t_all = time.time()
for seed in SEEDS:
    pf = f'cx_seed{seed}_{PIPELINE_VERSION}.pkl'
    cfg_fp = json.dumps([best_params, seed, N_SINGLE, N_WARM, BATCHES, THREADS, TIER1_MS, TIER2_MS, SMOKE_TEST, HW['cpu'], 'nb13-v1'], sort_keys=True, default=str)
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') == cfg_fp: RES[seed] = saved['res']; print(f'seed {seed}: result file found -- skipped'); continue
    print(f'=== seed {seed}', flush=True); r = {}; rng = np.random.RandomState(seed)
    # ---- models
    t0 = time.perf_counter(); gpu_m = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); gpu_m.fit(X_train, y_train, verbose=False); r['fit_gpu_s'] = time.perf_counter() - t0
    tmp = os.path.join(tempfile.gettempdir(), f'cx_std_{seed}_{os.getpid()}.json'); gpu_m.save_model(tmp); std = cpu_copy(tmp)
    rob_path = f'xgb_robust_{PIPELINE_VERSION}_seed{seed}.json'; rss0 = rss_mb(); rob = cpu_copy(rob_path); gate_models, tau = load_gate(seed); gnp = GateNP(gate_models, tau); r['rss_delta_mb'] = rss_mb() - rss0
    GATE_CUR = gate_models; mlp_t = SurrogateMLP(N_FEAT).eval(); mlp_np = MlpNP(mlp_t); b_std, b_rob = std.get_booster(), rob.get_booster()
    if seed == SEEDS[0] and not SMOKE_TEST:
        t0 = time.perf_counter(); cpu_fit = xgb.XGBClassifier(**{**best_params, **COMMON, 'device': 'cpu', 'n_jobs': -1}, random_state=seed); cpu_fit.fit(X_train, y_train, verbose=False); r['fit_cpu_s'] = time.perf_counter() - t0; del cpu_fit
    # ---- sizes
    r['size'] = {'std_json_kb': os.path.getsize(tmp) / 1024, 'rob_json_kb': os.path.getsize(rob_path) / 1024}
    ubj = tmp.replace('.json', '.ubj'); b_rob.save_model(ubj); r['size']['rob_ubj_kb'] = os.path.getsize(ubj) / 1024; os.remove(ubj)
    tg = os.path.join(tempfile.gettempdir(), f'cx_gate_{seed}.pt'); torch.save([m.state_dict() for m in gate_models], tg); r['size']['gate_pt_kb'] = os.path.getsize(tg) / 1024; os.remove(tg)
    df = b_rob.trees_to_dataframe(); r['size'].update({'rob_rounds': int(b_rob.num_boosted_rounds()), 'rob_trees': int(df['Tree'].nunique()), 'rob_nodes': int(len(df)), 'rob_leaves': int((df['Feature'] == 'Leaf').sum()), 'gate_params': int(sum(p.numel() for m in gate_models for p in m.parameters())), 'mlp_params': int(sum(p.numel() for p in mlp_t.parameters()))})
    # ---- correctness checks (first 1000 sample rows)
    Xc = np.vstack(Xs[:1000]); e_t = np.array([gate_torch_err(gate_models, xt) for xt in Xs_t[:1000]]); e_n = np.array([gnp.err1(x) for x in Xs_1d[:1000]]); e_b = gnp.err_batch(Xc)
    p_sk = rob.predict(Xc); p_bo = np.argmax(b_rob.inplace_predict(Xc), 1)
    with torch.no_grad(): pm_t = mlp_t(torch.from_numpy(Xc)).argmax(1).numpy()
    pm_n = np.array([mlp_np.pred1(x) for x in Xs_1d[:1000]])
    r['check'] = {'gate_np_vs_torch_max_abs_rel': float(np.max(np.abs(e_n - e_t) / np.maximum(e_t, 1e-12))), 'gate_batch_vs_single_max_rel': float(np.max(np.abs(e_b - e_n) / np.maximum(e_n, 1e-12))), 'flag_agree': float(((e_t > tau) == (e_n > tau)).mean()),
                  'xgb_sklearn_vs_booster_agree': float((p_sk == p_bo).mean()), 'mlp_np_vs_torch_agree': float((pm_t == pm_n).mean()), 'flagged_share': float((e_t > tau).mean())}
    assert r['check']['gate_np_vs_torch_max_abs_rel'] < 1e-3 and r['check']['xgb_sklearn_vs_booster_agree'] == 1.0, f'equivalence check failed: {r["check"]}'
    # ---- conformal helpers (placeholder calibration arrays of the true size)
    q_split = 0.5; A = [np.sort(rng.rand(N_CAL // N_CV)).astype(np.float32) for _ in range(N_CV)]
    cv_boosters = []
    for _ in range(N_CV):
        c = cpu_copy(rob_path); cv_boosters.append(c.get_booster())
    # ---- single-row latency
    r['lat'] = {}
    for k in THREADS:
        with limit_threads(k):
            set_xgb_threads(std, k); set_xgb_threads(rob, k)
            for b in cv_boosters: b.set_param({'nthread': k})
            def pipe_np(x1d, x2d=None):
                return -1 if gnp.err1(x1d) > gnp.tau else int(np.argmax(b_rob.inplace_predict(x1d[None])))
            def pipe_torch(xt):
                return -1 if gate_torch_err(gate_models, xt) > tau else int(np.argmax(b_rob.inplace_predict(xt.numpy())))
            def cp_split(x): p = b_rob.inplace_predict(x); return (1 - p) <= q_split
            def cp_cv(x):
                cnt = np.zeros(6, np.int64)
                for kk, b in enumerate(cv_boosters): cnt += np.searchsorted(A[kk], 1.0 - b.inplace_predict(x)[0], side='left')
                return cnt < 0.9 * (N_CAL + 1)
            def shap1(x): return b_rob.predict(xgb.DMatrix(x), pred_contribs=True)
            def mlp_torch(xt):
                with torch.no_grad(): return int(mlp_t(xt).argmax(1))
            comps = {'xgb_std (sklearn API)': (lambda x: std.predict(x), Xs), 'xgb_std (lean Booster)': (lambda x: np.argmax(b_std.inplace_predict(x), 1), Xs), 'xgb_robust (sklearn API)': (lambda x: rob.predict(x), Xs), 'xgb_robust (lean Booster)': (lambda x: np.argmax(b_rob.inplace_predict(x), 1), Xs),
                     'AE gate (PyTorch)': (lambda xt: gate_torch_err(gate_models, xt), Xs_t), 'AE gate (NumPy)': (gnp.err1, Xs_1d), 'pipeline gate+robust (PyTorch gate)': (pipe_torch, Xs_t), 'pipeline gate+robust (NumPy gate)': (pipe_np, Xs_1d),
                     'TreeSHAP, one row (robust)': (shap1, Xs), 'split-conformal set': (cp_split, Xs), 'CV+ conformal set (6 models)': (cp_cv, Xs), 'federated MLP (PyTorch)': (mlp_torch, Xs_t), 'federated MLP (NumPy)': (mlp_np.pred1, Xs_1d)}
            for name, (fn, xs) in comps.items():
                t = time_calls(fn, xs); r['lat'][(name, k)] = {**lat_stats(t, rng), 'times': t.astype(np.float32)}
            print(f'  threads {k}: ' + ' | '.join(f'{n.split(" (")[0][:14]} {r["lat"][(n, k)]["median"]:.3f}' for n in ['xgb_robust (lean Booster)', 'AE gate (NumPy)', 'pipeline gate+robust (NumPy gate)']) + ' ms median', flush=True)
    # ---- batch throughput (CPU all threads, GPU if present)
    r['thr'] = {}; Xb_all = X_test
    def run_batch(fn, bs, reps):
        Xb = np.ascontiguousarray(Xb_all[:bs]); ts = []
        fn(Xb)
        for _ in range(reps): t0 = time.perf_counter(); fn(Xb); ts.append(time.perf_counter() - t0)
        return float(np.median(ts)) / bs * 1e3                                       # ms per row
    with limit_threads(N_ALL):
        set_xgb_threads(rob, N_ALL)
        for bs in BATCHES:
            reps = 5 if bs >= 16384 else 20
            r['thr'][('xgb_robust CPU', bs)] = run_batch(lambda X: b_rob.inplace_predict(X), bs, reps); r['thr'][('AE gate NumPy CPU', bs)] = run_batch(gnp.err_batch, bs, reps)
            r['thr'][('AE gate PyTorch CPU', bs)] = run_batch(gate_torch_batch, bs, reps)
            r['thr'][('federated MLP PyTorch CPU', bs)] = run_batch(lambda X: mlp_t(torch.from_numpy(X)), bs, reps)
    if torch.cuda.is_available():
        gpu_b = gpu_m.get_booster(); gm = [m.to('cuda') for m in gate_models]; mg = SurrogateMLP(N_FEAT).to('cuda').eval()
        def gerr(X):
            xt = torch.from_numpy(X).to('cuda')
            with torch.no_grad(): e = torch.stack([((m(xt) - xt) ** 2).mean(1) for m in gm]).mean(0).cpu()
            return e
        def gmlp(X):
            with torch.no_grad(): return mg(torch.from_numpy(X).to('cuda')).argmax(1).cpu()
        for bs in BATCHES:
            reps = 5 if bs >= 16384 else 20
            r['thr'][('xgb_std GPU', bs)] = run_batch(lambda X: gpu_b.inplace_predict(X), bs, reps); r['thr'][('AE gate PyTorch GPU', bs)] = run_batch(gerr, bs, reps); r['thr'][('federated MLP PyTorch GPU', bs)] = run_batch(gmlp, bs, reps)
        xs_g = Xs[:300]; t = time_calls(lambda x: gpu_b.inplace_predict(x), xs_g, 50); r['lat'][('xgb_std GPU single row', 'gpu')] = {**lat_stats(t, rng), 'times': t.astype(np.float32)}
        xs_gt = [x.to('cuda') for x in Xs_t[:300]]; t = time_calls(lambda xt: gate_torch_err(gm, xt), xs_gt, 50); r['lat'][('AE gate PyTorch GPU single row', 'gpu')] = {**lat_stats(t, rng), 'times': t.astype(np.float32)}
    os.remove(tmp); RES[seed] = r; joblib.dump({'fp': cfg_fp, 'res': r}, pf)
    print(f'  seed {seed} done | GPU fit {r["fit_gpu_s"]:.0f}s' + (f', CPU fit {r["fit_cpu_s"]:.0f}s' if 'fit_cpu_s' in r else '') + f' | checks {r["check"]}', flush=True)
print(f'all seeds done in {(time.time()-t_all)/60:.1f} min')

## 4. Gate training time (NB7 loop, seed 42)

In [ ]:
def train_gate_timed(device, seed, folds):
    Xb = np.ascontiguousarray(X_train[y_train == 0]); kf = list(KFold(N_FOLDS, shuffle=True, random_state=seed).split(Xb))[:folds]; t0 = time.perf_counter()
    for k, (tr, ho) in enumerate(kf):
        torch.manual_seed(seed * 10 + k); m = AE(N_FEAT).to(device); opt = torch.optim.Adam(m.parameters(), lr=1e-3); xt = torch.from_numpy(Xb[tr]).float().to(device); g = torch.Generator().manual_seed(seed * 10 + k)
        for ep in range(2 if SMOKE_TEST else GATE_EPOCHS):
            m.train(); perm = torch.randperm(len(xt), generator=g).to(device)
            for s in range(0, len(xt), 256):
                xb = xt[perm[s:s + 256]]; opt.zero_grad(); ((m(xb) - xb) ** 2).mean().backward(); opt.step()
    if device.type == 'cuda': torch.cuda.synchronize()
    return time.perf_counter() - t0
GTRAIN = {}
if DEVICE.type == 'cuda': GTRAIN['gate, 5 folds x 30 epochs, GPU'] = train_gate_timed(DEVICE, SEEDS[0], N_FOLDS)
with limit_threads(N_ALL): GTRAIN['gate, 1 fold x 30 epochs, CPU (all threads)'] = train_gate_timed(torch.device('cpu'), SEEDS[0], 1)
GTRAIN['gate, 5 folds on CPU = 5 x one fold (estimate)'] = 5 * GTRAIN['gate, 1 fold x 30 epochs, CPU (all threads)']
print({k: round(v, 1) for k, v in GTRAIN.items()})

## 5. Statistics helpers

In [ ]:
def ptest(a, b):
    d = np.asarray(a, float) - np.asarray(b, float)
    if len(d) < 2: return float('nan')
    return 1.0 if d.std(ddof=1) == 0 and d.mean() == 0 else (float('nan') if d.std(ddof=1) == 0 else float(stats.ttest_rel(a, b).pvalue))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rk, i in enumerate(o): run = max(run, (len(p) - rk) * p[i]); adj[i] = min(1.0, run)
    return adj
def dz(a, b):
    d = np.asarray(a, float) - np.asarray(b, float); s = d.std(ddof=1) if len(d) > 1 else 0
    return float(d.mean() / s) if s > 0 else float('nan')
def one_sample_p(x, mu):
    x = np.asarray(x, float)
    if len(x) < 2 or x.std(ddof=1) == 0: return float('nan')
    return float(stats.ttest_1samp(x, mu).pvalue)
def fp(p): return '< 0.0001' if (p == p and p < 5e-5) else (f'{p:.4f}' if p == p else 'n/a')
ms = lambda x, d=3: f'{np.mean(x):.{d}f} +/- {(np.std(x, ddof=1) if len(x) > 1 else 0):.{d}f}'
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); T = {}

## 6. Results

In [ ]:
T = {}; pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
T['hardware'] = pd.Series(HW).rename('value').reset_index().rename(columns={'index': 'item'}); print('6a. Hardware and software (every number below belongs to this machine)'); print(T['hardware'].to_string(index=False))
LK = lambda name, k, f: np.array([RES[s]['lat'][(name, k)][f] for s in SEEDS])
names = [n for (n, k) in RES[SEEDS[0]]['lat'] if k == THREADS[0]]
print('\n6b. Single-row latency in ms (per call, Python overhead included). mean +/- std over 5 seeds of the per-seed statistic; CI = bootstrap over calls (indicative), mean of seeds. Under-bound = share of calls below the bound.')
rows = []
for k in THREADS:
    for n in names:
        rows.append({'component': n, 'threads': k, 'median': ms(LK(n, k, 'median'), 3), 'median CI': f'[{np.mean([RES[s]["lat"][(n,k)]["median_ci"][0] for s in SEEDS]):.3f}, {np.mean([RES[s]["lat"][(n,k)]["median_ci"][1] for s in SEEDS]):.3f}]', 'mean': ms(LK(n, k, 'mean'), 3), 'P95': ms(LK(n, k, 'p95'), 3),
                     'P99': ms(LK(n, k, 'p99'), 3), 'P99 CI': f'[{np.mean([RES[s]["lat"][(n,k)]["p99_ci"][0] for s in SEEDS]):.3f}, {np.mean([RES[s]["lat"][(n,k)]["p99_ci"][1] for s in SEEDS]):.3f}]', 'max': ms(LK(n, k, 'max'), 2),
                     f'< {TIER1_MS:g} ms': ms(LK(n, k, 'under10'), 4), f'< {TIER2_MS:g} ms': ms(LK(n, k, 'under100'), 4)})
for s_key in [('xgb_std GPU single row', 'gpu'), ('AE gate PyTorch GPU single row', 'gpu')]:
    if s_key in RES[SEEDS[0]]['lat']:
        n, k = s_key; rows.append({'component': n, 'threads': 'GPU', 'median': ms(LK(n, k, 'median'), 3), 'median CI': '', 'mean': ms(LK(n, k, 'mean'), 3), 'P95': ms(LK(n, k, 'p95'), 3), 'P99': ms(LK(n, k, 'p99'), 3), 'P99 CI': '', 'max': ms(LK(n, k, 'max'), 2), f'< {TIER1_MS:g} ms': ms(LK(n, k, 'under10'), 4), f'< {TIER2_MS:g} ms': ms(LK(n, k, 'under100'), 4)})
T['latency_single'] = pd.DataFrame(rows); print(T['latency_single'].to_string(index=False))

print('\n6c. Tier check at 1 thread (an edge-class setting) using P99 and its bootstrap upper bound; bounds are the thesis\'s assumed requirements')
rows = []
for n in ['xgb_robust (sklearn API)', 'xgb_robust (lean Booster)', 'AE gate (PyTorch)', 'AE gate (NumPy)', 'pipeline gate+robust (PyTorch gate)', 'pipeline gate+robust (NumPy gate)', 'TreeSHAP, one row (robust)', 'CV+ conformal set (6 models)']:
    p99 = LK(n, 1, 'p99'); up = np.mean([RES[s]['lat'][(n, 1)]['p99_ci'][1] for s in SEEDS])
    rows.append({'component': n, 'P99 ms': ms(p99, 3), 'P99 upper CI': f'{up:.3f}', 'Tier 1 (< 10 ms)': 'meets' if up < TIER1_MS else ('P99 meets, CI does not' if p99.mean() < TIER1_MS else 'misses'), 'Tier 2 (< 100 ms)': 'meets' if up < TIER2_MS else ('P99 meets, CI does not' if p99.mean() < TIER2_MS else 'misses'), 'worst single call (max over seeds), ms': f'{LK(n, 1, "max").max():.2f}'})
T['tier'] = pd.DataFrame(rows); print(T['tier'].to_string(index=False))
print('   v4 (Colab T4): XGBoost alone 6.26 ms, gated pipeline 13.2 ms mean / 53.1 ms P99. Compare against the rows above only as a statement about this machine versus that one.')

print('\n6d. Paired comparisons of per-seed median latency at 1 thread (difference = first minus second, ms; Holm over this table; dz = paired effect size)')
PAIRS = [('xgb_robust (sklearn API)', 'xgb_robust (lean Booster)', 'sklearn wrapper overhead'), ('xgb_robust (lean Booster)', 'xgb_std (lean Booster)', 'robust vs standard model'), ('AE gate (PyTorch)', 'AE gate (NumPy)', 'PyTorch vs NumPy gate'),
         ('pipeline gate+robust (NumPy gate)', 'xgb_robust (lean Booster)', 'cost of adding the gate (NumPy)'), ('pipeline gate+robust (PyTorch gate)', 'pipeline gate+robust (NumPy gate)', 'pipeline PyTorch vs NumPy gate'),
         ('CV+ conformal set (6 models)', 'split-conformal set', 'CV+ vs split conformal'), ('TreeSHAP, one row (robust)', 'xgb_robust (lean Booster)', 'explaining one row vs predicting it'), ('federated MLP (PyTorch)', 'federated MLP (NumPy)', 'MLP PyTorch vs NumPy')]
pl = [ptest(LK(a, 1, 'median'), LK(b, 1, 'median')) for a, b, _ in PAIRS]; ph = holm(pl)
T['compare'] = pd.DataFrame([{'comparison': lab, 'first': a, 'second': b, 'difference (ms)': f'{(LK(a, 1, "median") - LK(b, 1, "median")).mean():+.4f}', 'ratio': f'{(LK(a, 1, "median") / LK(b, 1, "median")).mean():.2f}x', 'dz': f'{dz(LK(a, 1, "median"), LK(b, 1, "median")):.2f}', 'p (Holm)': fp(ph[i])} for i, (a, b, lab) in enumerate(PAIRS)]); print(T['compare'].to_string(index=False))
print('   Seeds change the models; the machine noise between runs is mixed into the spread. A small p here means the difference exceeds that combined noise, not that it is a universal constant.')

print('\n6e. Drift check: median latency (ms) in %d consecutive blocks of the 1-thread run (a large first block means the machine was still warming up or another process interfered)' % N_BLOCKS)
rows = [{'component': n, **{f'block {j + 1}': ms(np.array([RES[s]['lat'][(n, 1)]['block_medians'][j] for s in SEEDS]), 3) for j in range(N_BLOCKS)}} for n in names]
T['drift'] = pd.DataFrame(rows); print(T['drift'].to_string(index=False))

In [ ]:
print('6f. Batch throughput: milliseconds per row (median of repeats, mean over seeds), CPU with all %d threads; GPU if present. Per-call Python overhead is amortised over the batch.' % N_ALL)
keys = sorted({k[0] for k in RES[SEEDS[0]]['thr']}); rows = []
for kk in keys: rows.append({'component': kk, **{f'batch {bs}': f'{np.mean([RES[s]["thr"][(kk, bs)] for s in SEEDS]):.5f}' for bs in BATCHES}, 'rows/s at largest batch': f'{1e3 / np.mean([RES[s]["thr"][(kk, BATCHES[-1])] for s in SEEDS]):,.0f}'})
T['throughput'] = pd.DataFrame(rows); print(T['throughput'].to_string(index=False))

print('\n6g. Size and complexity')
S = lambda k: np.array([RES[s]['size'][k] for s in SEEDS])
macs = {'AE gate (5 AEs)': 5 * (N_FEAT * GATE_HIDDEN + GATE_HIDDEN * GATE_BOTTLENECK + GATE_BOTTLENECK * GATE_HIDDEN + GATE_HIDDEN * N_FEAT), 'federated MLP': N_FEAT * 64 + 64 * 32 + 32 * 6}
md_ = best_params.get('max_depth', 6); rounds = int(S('rob_rounds')[0])
T['size'] = pd.DataFrame([
    {'item': 'robust XGBoost, JSON file (KB)', 'value': ms(S('rob_json_kb'), 0)}, {'item': 'robust XGBoost, UBJ file (KB)', 'value': ms(S('rob_ubj_kb'), 0)}, {'item': 'standard XGBoost, JSON file (KB)', 'value': ms(S('std_json_kb'), 0)},
    {'item': 'boosting rounds / trees / nodes / leaves (robust)', 'value': f'{rounds} / {int(S("rob_trees")[0])} / {int(S("rob_nodes")[0]):,} / {int(S("rob_leaves")[0]):,}'}, {'item': 'XGBoost comparisons per row (upper bound = trees x max depth)', 'value': f'{int(S("rob_trees")[0]) * md_:,}'},
    {'item': 'AE gate state files (KB), 5 AEs', 'value': ms(S('gate_pt_kb'), 1)}, {'item': 'AE gate parameters (5 AEs)', 'value': f'{int(S("gate_params")[0]):,}'}, {'item': 'AE gate multiply-adds per row', 'value': f'{macs["AE gate (5 AEs)"]:,}'},
    {'item': 'federated MLP parameters / multiply-adds', 'value': f'{int(S("mlp_params")[0]):,} / {macs["federated MLP"]:,}'}, {'item': 'process memory growth when loading robust model + gate (MB, RSS)', 'value': ms(np.array([RES[s]['rss_delta_mb'] for s in SEEDS]), 1)}]); print(T['size'].to_string(index=False))
print('   Total deployed pipeline file size = robust UBJ/JSON + gate state; the v4 figure was 9.31 MB.')

print('\n6h. Training time on this machine')
rows = [{'what': 'XGBoost fit, standard tuned model, GPU (per seed)', 'seconds': ms(np.array([RES[s]['fit_gpu_s'] for s in SEEDS]), 1)}]
if 'fit_cpu_s' in RES[SEEDS[0]]: rows.append({'what': f'XGBoost fit, standard tuned model, CPU all threads (seed {SEEDS[0]} only)', 'seconds': f'{RES[SEEDS[0]]["fit_cpu_s"]:.1f}'})
rows += [{'what': k, 'seconds': f'{v:.1f}'} for k, v in GTRAIN.items()]
T['training'] = pd.DataFrame(rows); print(T['training'].to_string(index=False))
print('   Not included: NB5b adversarial training (attack generation) -- see its log. The v4 statement "retraining below 3 minutes" can be checked against the rows above only for the parts measured here.')

print('\n6i. Correctness checks (mean over seeds; the run stops if the NumPy gate or the lean XGBoost path disagrees)')
T['checks'] = pd.DataFrame([{'check': k, 'value': f'{np.mean([RES[s]["check"][k] for s in SEEDS]):.6g}', 'worst over seeds': f'{np.max([RES[s]["check"][k] for s in SEEDS]):.6g}' if 'agree' not in k else f'{np.min([RES[s]["check"][k] for s in SEEDS]):.6g}'} for k in RES[SEEDS[0]]['check']]); print(T['checks'].to_string(index=False))
print('   flagged_share = share of the 1,000 checked test rows the gate flags (the pipeline skips the classifier for them, so its mean latency is lower than gate + classifier).')

## 7. Figures and saved outputs

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(16, 11)); show = [n for n in names if 'SHAP' not in n]
for j, k in enumerate(THREADS):
    xs = np.arange(len(names)); ax[0, 0].bar(xs + (j - 0.5) * 0.4, [LK(n, k, 'median').mean() for n in names], 0.4, label=f'median, {k} thread(s)'); ax[0, 1].bar(xs + (j - 0.5) * 0.4, [LK(n, k, 'p99').mean() for n in names], 0.4, label=f'P99, {k} thread(s)')
for a_, t_ in [(ax[0, 0], 'Single-row median latency (ms)'), (ax[0, 1], 'Single-row P99 latency (ms)')]:
    a_.set_yscale('log'); a_.set_xticks(np.arange(len(names))); a_.set_xticklabels(names, rotation=80, fontsize=6); a_.set_title(t_); a_.legend(fontsize=8); a_.grid(alpha=.3, axis='y')
ax[0, 1].axhline(TIER1_MS, color='r', ls='--', lw=1); ax[0, 1].axhline(TIER2_MS, color='k', ls='--', lw=1)
for kk in keys: ax[1, 0].plot(BATCHES, [np.mean([RES[s]['thr'][(kk, bs)] for s in SEEDS]) for bs in BATCHES], marker='o', label=kk)
ax[1, 0].set_xscale('log'); ax[1, 0].set_yscale('log'); ax[1, 0].set_xlabel('batch size'); ax[1, 0].set_ylabel('ms per row'); ax[1, 0].set_title('Batch cost per row'); ax[1, 0].legend(fontsize=6); ax[1, 0].grid(alpha=.3)
for n in ['xgb_robust (lean Booster)', 'pipeline gate+robust (NumPy gate)', 'pipeline gate+robust (PyTorch gate)']: ax[1, 1].hist(np.concatenate([RES[s]['lat'][(n, 1)]['times'] for s in SEEDS]), bins=np.logspace(-2, 2, 80), alpha=0.5, label=n)
ax[1, 1].set_xscale('log'); ax[1, 1].set_xlabel('ms per call (1 thread, all seeds pooled)'); ax[1, 1].set_title('Latency distribution'); ax[1, 1].legend(fontsize=7); ax[1, 1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'cx_{PIPELINE_VERSION}.png', dpi=150); plt.show()
joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'tables': T, 'hardware': HW, 'gate_training': GTRAIN, 'per_seed': {s: RES[s] for s in SEEDS}, 'config': {'N_SINGLE': N_SINGLE, 'N_WARM': N_WARM, 'THREADS': THREADS, 'BATCHES': BATCHES}}, f'cx_results_{PIPELINE_VERSION}.pkl')
for k_, t_ in T.items(): t_.to_csv(f'cx_{k_}_{PIPELINE_VERSION}.csv', index=False)
json.dump({'notebook': 'NB13', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'xgboost': xgb.__version__, 'torch': torch.__version__, 'hardware': {k: str(v) for k, v in HW.items()}, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb13_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'cx_results_{PIPELINE_VERSION}.pkl', f'cx_{PIPELINE_VERSION}.png', f'nb13_run_manifest_{PIPELINE_VERSION}.json'] + [f'cx_{k_}_{PIPELINE_VERSION}.csv' for k_ in T]: print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)